How to install Hugging_face
``` python
#Step 1: Install the Transformers library
pip install transformers
# Step 2: Install a backend framework
pip install torch

In [2]:
import torch
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

print(classifier("I absolutely loved this movie!"))

ModuleNotFoundError: Could not import module 'pipeline'. Are this object's requirements defined correctly?

In [ ]:
classifier = pipeline("sentiment-analysis")
classifier("This restaurant was terrible.")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9994051456451416}]

In [ ]:
import transformers
print("Transformers version:", transformers.__version__)
import torch
print("PyTorch version:", torch.__version__)

Transformers version: 5.17.0
PyTorch version: 2.14.0+cu130


``` text 
SMILES : simplified molecular input line entry system
SMILES / text
      ↓
Tokenizer
      ↓
Token IDs
      ↓
Pretrained Transformer
      ↓
Hidden representations
      ↓
Embedding
      ↓
ML model / downstream task
``` 
| Your data/task                        | Model family to look for                   |
| ------------------------------------- | ------------------------------------------ |
| Normal English text                   | BERT, DistilBERT, RoBERTa                  |
| Scientific papers/text                | SciBERT                                    |
| Chemical SMILES                       | ChemBERTa and other molecular Transformers |
| Materials/crystal/MOF representations | Materials-specific pretrained models       |
| Text generation/chat                  | Generative models                          |


In [ ]:
# I give Hugging Face a model name, and Hugging Face automatically determines the correct Python model class
from transformers import AutoTokenizer, AutoModel
import torch
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModel.from_pretrained(model_name)
texts = [
    "Carbon dioxide adsorption in porous materials.",
    "Metal organic frameworks have high surface area.",
    "Benzene contains an aromatic ring."
]
text = texts[0]

tokens = tokenizer.tokenize(text)

print(tokens)
inputs = tokenizer(
    text,
    return_tensors="pt"
)

print(inputs)

with torch.no_grad():
    outputs = model(**inputs)
print(outputs.last_hidden_state.shape)
token_embeddings = outputs.last_hidden_state

sentence_embedding = token_embeddings.mean(dim=1)

print(sentence_embedding.shape)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['carbon', 'dioxide', 'ads', '##or', '##ption', 'in', 'por', '##ous', 'materials', '.']
{'input_ids': tensor([[  101,  6351, 14384, 14997,  2953, 16790,  1999, 18499,  3560,  4475,
          1012,   102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}
torch.Size([1, 12, 768])
torch.Size([1, 768])


``` text 
Text
 ↓
TF-IDF
 ↓
numbers
 ↓
Machine-learning model

A Transformer
Text
 ↓
Transformer
 ↓
numbers (embedding)
 ↓
Machine-learning model

A Transformer can be used as a feature extractor. It converts complicated input into numerical representations (embeddings), and those representations can then be used for tasks such as classification, regression, similarity, or prediction.

A model card: Documentation explaining what the model does, how it was trained, its limitations, and how to use it.

Model files: The actual weights and configuration files needed to run the model.

Usage statistics: Download counts, likes, and community engagement.

Code examples: Snippets showing how to load and use the model.

Models are organized by task. For RAG systems, the most relevant categories are:

| **Task** | **What It Does** | **Example Models** |
|---|---|---|
| Feature Extraction | Creates embeddings from text | all-MiniLM-L6-v2, bge-large |
| Question Answering | Extracts answers from context | distilbert-base-cased-distilled-squad |
| Text Generation | Generates text continuations | Llama, Mistral, Falcon |
| Text2Text Generation | Transforms input text to output text | T5, FLAN-T5 |
| Fill-Mask | Predicts masked words | BERT, RoBERTa |

For our RAG system, we will use:

Embedding models (Feature Extraction) to convert text into searchable vectors
Question Answering models to extract answers from retrieved context

Question Answering Pipeline
For RAG systems, the question-answering pipeline is particularly relevant. It takes a question and a context, then extracts the answer from the context:

``` python

from transformers import pipeline

# Create QA pipeline
qa_model = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)

# Provide context and question
context = """
The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France.
It was constructed from 1887 to 1889 as the centerpiece of the 1889 World's Fair.
The tower is 330 meters tall and was the tallest man-made structure in the world
until the Chrysler Building was completed in 1930.
"""

question = "How tall is the Eiffel Tower?"

# Get answer
result = qa_model(question=question, context=context)
print(f"Answer: {result['answer']}")
print(f"Confidence: {result['score']:.2%}")
```